# Planning Baselines

This notebook compares standard geometric A* with a rule-based
safety-aware expert A* planner.

In [ ]:
import sys
from pathlib import Path

import numpy as np

In [ ]:
def find_project_root() -> Path:
    """Find the project directory containing the src folder."""
    current_directory = Path.cwd().resolve()

    for candidate in [
        current_directory,
        current_directory.parent,
    ]:
        if (candidate / "src").is_dir():
            return candidate

    raise RuntimeError(
        "Could not find the project root containing src/."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print("Project root:", PROJECT_ROOT)

In [ ]:
from src.cost_maps import build_safety_cost_maps
from src.planners import (
    astar_search,
    cost_aware_astar,
    validate_path,
)
from src.scene_generator import generate_driving_scene
from src.visualization import (
    plot_path_comparison,
    plot_safety_cost_maps,
)

In [ ]:
scene = generate_driving_scene(seed=4)

standard_result = astar_search(
    free_space=scene.free_space,
    start=scene.start,
    goal=scene.goal,
)

safety_cost_maps = build_safety_cost_maps(
    scene
)

expert_result = cost_aware_astar(
    free_space=scene.free_space,
    traversal_cost=safety_cost_maps.traversal_cost,
    start=scene.start,
    goal=scene.goal,
)

print("Standard A* success:", standard_result.success)
print("Expert A* success:", expert_result.success)

print(
    "Standard path valid:",
    validate_path(
        scene.free_space,
        standard_result.path,
        scene.start,
        scene.goal,
    ),
)

print(
    "Expert path valid:",
    validate_path(
        scene.free_space,
        expert_result.path,
        scene.start,
        scene.goal,
    ),
)

In [ ]:
plot_safety_cost_maps(
    scene,
    safety_cost_maps,
)

In [ ]:
plot_path_comparison(
    scene,
    standard_result,
    expert_result,
)